# 05 · Reference baselines (Stage A: E1–E5)

**Goal:** establish the reference points every continual-learning result is measured against (`docs/RESEARCH_PLAN.md` §3, Stage A).

| ID | Experiment | Question |
|---|---|---|
| E1 | Within-group chronological split, one model on all groups | Later-life error when every group is seen |
| E2 | RPT-boundary split (no RPT interval shared across splits) | Error on SOH states never seen in training (D4) |
| E3 | Leave-one-group-out, tested on the held-out group's E1 test period | Unseen-group error (RQ1, H1) |
| E4 | Joint MLP training = E1 with the MLP backbone | Upper bound for continual learning |
| E5 | E1 after relabelling same-day cycles with the `"end"` convention | Sensitivity to the date-only rule (D3) |

Two models: Ridge (the notebook 04 baseline, alpha chosen on validation) and the MLP backbone used for continual learning, so Stage A and Stage C are directly comparable. Primary metrics are RMSE and MAE in SOH percentage points (D8).

Set `EXPERIMENT_PROFILE=smoke` for a fast code check (few epochs); results from that profile are not for reporting.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})

from src import evaluation as ev
from src import experiments as X

PROFILE = os.environ.get("EXPERIMENT_PROFILE", "paper")
CONFIG = {
    "profile": PROFILE,
    "models": ["ridge", "mlp"],
    "seed": 0,
    "train_frac": 0.70,
    "val_frac": 0.15,
    "mlp_kwargs": {"epochs": 200, "patience": 20} if PROFILE == "paper" else {"epochs": 10, "patience": 3},
}
print(CONFIG)

## 2. Load data

`ml_training_dataset.csv` is the supervised input; `final_battery_health_dataset.csv` is needed for the E5 relabelling.

In [ ]:
dp.check_data_files(DATA_ROOT, required=["ml_training_dataset.csv", "final_battery_health_dataset.csv"])
data = dp.read_dataset(DATA_ROOT / "ml_training_dataset.csv")
health = dp.read_dataset(DATA_ROOT / "final_battery_health_dataset.csv")
data[dp.GROUP_COL] = data[dp.GROUP_COL].astype(str)
print("rows:", len(data), "| groups:", dp.check_battery_groups(data[dp.GROUP_COL]))

## 3. Check the RPT-boundary split before using it

E2 needs at least three RPT intervals per battery and RPT labels in time order. Any failure here stops the notebook rather than silently dropping a battery.

In [ ]:
from src import protocols as P

rpt_split = P.rpt_boundary_split(data, CONFIG["train_frac"], CONFIG["val_frac"])
rpt_check = P.check_rpt_boundary(rpt_split)
display(rpt_check)
assert rpt_check["ok"].all(), "RPT-boundary split failed for some battery; see table above"

## 4. Run E1–E5

In [ ]:
results, details = X.reference_baselines(
    data, health, models=CONFIG["models"], seed=CONFIG["seed"],
    train_frac=CONFIG["train_frac"], val_frac=CONFIG["val_frac"], mlp_kwargs=CONFIG["mlp_kwargs"],
)
print(f"E5 relabelling: {details['E5_labels_changed']} labels changed, {details['E5_rows_dropped']} rows dropped (before first RPT under 'end')")
groups = [g for g in dp.BATTERY_GROUPS if g in set(results[dp.GROUP_COL])]
rmse = results.pivot_table(index=dp.GROUP_COL, columns=["model", "experiment"], values="RMSE").reindex(groups)
pooled = results.groupby(["model", "experiment"])[["RMSE", "MAE", "MAPE"]].mean().round(3)
display(rmse.round(3))
print("Mean over groups (each group weighted equally):")
display(pooled)

## 5. Per-group comparison

Test RMSE per battery group for the three main settings. Higher bars under E2 and E3 than E1 indicate how much of the E1 score comes from interpolating within known SOH states or from having seen the group.

In [ ]:
EXP_COLORS = {"E1_within_group": "#2a78d6", "E2_rpt_boundary": "#eb6834", "E3_leave_one_group_out": "#1baf7a"}
EXP_LABELS = {"E1_within_group": "E1 within-group", "E2_rpt_boundary": "E2 RPT-boundary", "E3_leave_one_group_out": "E3 unseen group"}
fig, axes = plt.subplots(1, len(CONFIG["models"]), figsize=(11, 3.8), sharey=True)
axes = np.atleast_1d(axes)
width = 0.26
x = np.arange(len(groups))
for ax, model in zip(axes, CONFIG["models"]):
    for k, exp in enumerate(EXP_COLORS):
        vals = rmse[(model, exp)].to_numpy()
        ax.bar(x + (k - 1) * width, vals, width=width - 0.03, color=EXP_COLORS[exp], label=EXP_LABELS[exp])
    ax.set_xticks(x, groups)
    ax.set_title(f"{model.upper() if model == 'mlp' else model.capitalize()}", loc="left")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Test RMSE (SOH percentage points)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.suptitle("Reference baselines: test RMSE per battery group")
plt.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

## 6. H1: is unseen-group error higher than within-group error?

Paired comparison over battery groups of E3 − E1 RMSE (positive = worse on an unseen group). With six groups the smallest attainable two-sided Wilcoxon p-value is 0.031, so the effect size and per-group differences matter as much as the p-value.

In [ ]:
h1_rows = []
for model in CONFIG["models"]:
    e1 = rmse[(model, "E1_within_group")]
    e3 = rmse[(model, "E3_leave_one_group_out")]
    e2 = rmse[(model, "E2_rpt_boundary")]
    h1_rows.append({"model": model, "comparison": "E3 - E1", **ev.paired_comparison(e3.to_numpy(), e1.to_numpy())})
    h1_rows.append({"model": model, "comparison": "E2 - E1", **ev.paired_comparison(e2.to_numpy(), e1.to_numpy())})
h1 = pd.DataFrame(h1_rows)
display(h1[["model", "comparison", "n", "mean_diff", "ci_low", "ci_high", "p_value", "rank_biserial"]].round(4))

## 7. E5: sensitivity to the date-only convention

E1 vs. E5 per model. E5 labels differ from E1 on same-day cycles, so this compares the whole pipeline under each convention, not identical targets.

In [ ]:
e5 = pd.DataFrame({
    (m, k): rmse[(m, e)] for m in CONFIG["models"] for k, e in [("E1 (start)", "E1_within_group"), ("E5 (end)", "E5_date_only_end")]
})
for m in CONFIG["models"]:
    e5[(m, "difference")] = e5[(m, "E5 (end)")] - e5[(m, "E1 (start)")]
e5.round(4)

## 8. Save results

In [ ]:
out = X.write_results(RESULTS_DIR / "reference_baselines.json", {
    "config": CONFIG, "details": details, "results": results, "h1": h1,
}, REPO_ROOT)
print("wrote", out.relative_to(REPO_ROOT))
if PROFILE != "paper":
    print("NOTE: smoke profile — these numbers are for code checks only.")

## 9. Interpretation checklist

* **E2 vs. E1:** a large gap means E1 benefits from test rows sharing RPT intervals with training (D4); E2 is the stricter number for the paper.
* **E3 vs. E1 (H1):** the gap is the cost of heterogeneity that continual learning must address. Report per group; one outlying group can dominate the mean.
* **E4 (E1 with the MLP):** the upper bound used in notebook 06.
* **E5:** if the E1–E5 differences are small relative to the E1 errors, conclusions do not depend on the date-only convention (D3).
* All results are within one handover of six groups; see the limitations in `docs/RESEARCH_PLAN.md` §2.